# Global Research Inequality: Who Gets to Do Science?

Science thrives on diverse perspectives — but access to research careers
is far from equal across the world. This notebook explores the global
distribution of R&D researchers using data from UNESCO and the World Bank,
covering 180+ countries from 1996 to 2023.

The data tells a striking story: a small number of high-income countries
concentrate the vast majority of the world's researchers, while large parts
of Africa, Southeast Asia, and Latin America remain severely underrepresented
in global science.

Understanding this inequality is the first step toward addressing it.

**Data source:** UNESCO UIS Stat Bulk Data Download Service, via World Bank
(2026), processed by Our World in Data.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("researchers-in-rd-per-million-people.csv")

In [3]:
df.columns = ["country", "code", "year", "researchers_per_million"]

print(f"Shape: {df.shape}")
print(f"Years: {df['year'].min()} – {df['year'].max()}")
print(f"Countries: {df['country'].nunique()}")
df.head(10)

Shape: (2076, 4)
Years: 1996 – 2023
Countries: 155


,country,code,year,researchers_per_million
0,Albania,ALB,2008,155.23569
1,Algeria,DZA,2005,170.18968
2,Algeria,DZA,2017,821.66516
3,American Samoa,ASM,2002,87.58715
4,American Samoa,ASM,2003,420.77948
5,American Samoa,ASM,2004,158.16112
6,American Samoa,ASM,2005,105.76042
7,Angola,AGO,2011,48.42677
8,Angola,AGO,2016,18.93579
9,Argentina,ARG,1997,694.15510


##Data Quality Check
Before moving on I check if the data has been properly cleaned.

In [4]:
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print(f"Duplicates: {df.duplicated().sum()}")

print("\n=== NEGATIVE OR ZERO VALUES ===")
print(f"Zero or negative: {(df['researchers_per_million'] <= 0).sum()}")

print("\n=== SUSPICIOUSLY HIGH VALUES ===")
print(df.nlargest(10, "researchers_per_million")[
    ["country", "year", "researchers_per_million"]
])

print("\n=== ENTRIES WITHOUT ISO CODE ===")
print(df[df["code"].isnull() | (df["code"] == "")][
    ["country", "code"]
].drop_duplicates())

print("\n=== NON-COUNTRY ENTITIES (aggregates/regions) ===")
print(df[df["code"].str.len() != 3][
    ["country", "code"]
].drop_duplicates())

=== MISSING VALUES ===
country                    0
code                       0
year                       0
researchers_per_million    0
dtype: int64

=== DUPLICATE ROWS ===
Duplicates: 0

=== NEGATIVE OR ZERO VALUES ===
Zero or negative: 0

=== SUSPICIOUSLY HIGH VALUES ===
            country  year  researchers_per_million
1002  Liechtenstein  2019               20047.0470
1003  Liechtenstein  2023               18129.7520
1730    South Korea  2022                9434.7620
1729    South Korea  2021                9071.4500
393         Denmark  2022                8735.6010
1794         Sweden  2022                8623.7460
1728    South Korea  2020                8620.0090
1727    South Korea  2019                8328.9670
1793         Sweden  2021                8159.7970
568         Finland  2022                8073.1455

=== ENTRIES WITHOUT ISO CODE ===
Empty DataFrame
Columns: [country, code]
Index: []

=== NON-COUNTRY ENTITIES (aggregates/regions) ===
                          

Clean data — no missing values, no duplicates, no negatives. Good foundation.

11 non-country entities — these need to be filtered out before building country-level visualizations, but we keep them separately as they're useful for the regional comparison line chart later.

In [5]:
# Non-country codes to exclude from country-level analysis
aggregate_codes = [
    "WB_EAP", "WB_ECA", "OWID_EU27", "OWID_HIC", "WB_LAC",
    "OWID_LMC", "WB_MENAP", "WB_NA", "WB_SA", "OWID_UMC", "OWID_WRL"
]

df_countries = df[~df["code"].isin(aggregate_codes)].copy()
df_aggregates = df[df["code"].isin(aggregate_codes)].copy()

print(f"Country rows: {len(df_countries)}")
print(f"Aggregate rows: {len(df_aggregates)}")
print(f"Countries: {df_countries['country'].nunique()}")
print(f"Aggregates: {df_aggregates['country'].nunique()}")

Country rows: 1887
Aggregate rows: 189
Countries: 144
Aggregates: 11


## A Note on Data Sparsity

Not every country reports R&D data every year. Some report only once or twice
over the entire period, while others have consistent annual data. This is common
in international datasets of this kind — data collection capacity itself reflects
the inequality we are measuring.

For visualizations requiring a single value per country, we use the **most
recent available data point** for each country.

In [6]:
latest = (
    df_countries.sort_values("year")
    .groupby(["country", "code"])
    .last()
    .reset_index()
)

print(f"Countries with data: {len(latest)}")
print(f"\nTop 5:")
print(latest.nlargest(5, "researchers_per_million")[
    ["country", "year", "researchers_per_million"]
].to_string(index=False))
print(f"\nBottom 5:")
print(latest.nsmallest(5, "researchers_per_million")[
    ["country", "year", "researchers_per_million"]
].to_string(index=False))

Countries with data: 144

Top 5:
      country  year  researchers_per_million
Liechtenstein  2023               18129.7520
  South Korea  2022                9434.7620
      Denmark  2022                8735.6010
       Sweden  2022                8623.7460
      Finland  2022                8073.1455

Bottom 5:
                     country  year  researchers_per_million
Democratic Republic of Congo  2015                 10.10515
                      Uganda  2023                 12.44448
                   Guatemala  2021                 14.52588
                        Laos  2002                 15.62884
                      Angola  2016                 18.93579


In [7]:
print("Global summary (latest data per country):")
print(latest["researchers_per_million"].describe().round(1))

median = latest["researchers_per_million"].median()
top10_mean = latest.nlargest(10, "researchers_per_million")[
    "researchers_per_million"
].mean()
bottom10_mean = latest.nsmallest(10, "researchers_per_million")[
    "researchers_per_million"
].mean()

print(f"\nGlobal median: {median:.0f} researchers per million")
print(f"Top 10 countries average: {top10_mean:.0f}")
print(f"Bottom 10 countries average: {bottom10_mean:.0f}")
print(f"Ratio top/bottom: {top10_mean/bottom10_mean:.0f}x")

Global summary (latest data per country):
count      144.0
mean      1931.8
std       2717.1
min         10.1
25%        131.6
50%        662.5
75%       2626.1
max      18129.8
Name: researchers_per_million, dtype: float64

Global median: 662 researchers per million
Top 10 countries average: 8881
Bottom 10 countries average: 18
Ratio top/bottom: 495x


## Key Findings from the Data

The data reveals a large global inequality in research capacity:

- The **global median** is 662 researchers per million people, meaning half
  of all countries fall below this threshold
- The **top 10 countries** average 8,881 researchers per million
- The **bottom 10 countries** average just 18 researchers per million
- This represents a **495x gap** between the most and least research-intensive
  countries in the world

The bottom of the distribution is dominated by Sub-Saharan African nations and
low-income countries in Asia and Latin America - the same regions that are
underrepresented in global scientific publishing, funding, and collaboration networks.

Note: some countries appear with data that is 10–20 years old, reflecting
inconsistent national reporting capacity, itself a symptom of the inequality
being measured.

## Visualizations

### 1. Where Are the World's Researchers?

The map below shows the most recent available data point for each country.
Hover over a country to see the exact value and the year it was recorded.
Grey countries have no data available.

In [12]:
fig_map = px.choropleth(
    latest,
    locations="code",
    color="researchers_per_million",
    hover_name="country",
    hover_data={"year": True, "researchers_per_million": ":.0f", "code": False},
    color_continuous_scale="Viridis",
    range_color=[0, 6000],  # cap at 6000 so mid-range countries show contrast
    labels={
        "researchers_per_million": "Researchers per million",
        "year": "Latest data year"
    },
    title="R&D Researchers per Million People (most recent data per country)"
)

fig_map.update_layout(
    geo=dict(showframe=False, showcoastlines=True,
    projection_type="natural earth"),
    coloraxis_colorbar=dict(
        title="Researchers<br>per million",
        tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000],
        ticktext=["0", "1,000", "2,000", "3,000", "4,000", "5,000", "6,000+"]
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    height=500
)

fig_map.show()

**Note on Liechtenstein:** Liechtenstein appears as an outlier with ~18,000
researchers per million people. This is not a data error — it reflects the
per-million scaling applied to a microstate with a population of ~38,000 and
a highly industrialized economy. A handful of
researchers represent a disproportionately large share of the population.

For this reason, microstates (population < 100,000) should be interpreted with
caution in per-capita metrics like this one.

The color scale is capped at 6,000 to preserve
contrast across the rest of the map; Liechtenstein and a few other high-performing
countries appear at the maximum color value.

### 2. Regional Trends Over Time

How has research capacity evolved across world regions since the late 1990s?
The chart below uses World Bank regional aggregates to show long-term trends.

Note: Europe and
Central Asia (including Russia) are combined into a single region in this
classification.

In [9]:
# Select the World Bank regional aggregates only
wb_regions = [
    "WB_EAP", "WB_ECA", "WB_LAC", "WB_MENAP", "WB_NA", "WB_SA"
]

region_labels = {
    "WB_EAP": "East Asia & Pacific",
    "WB_ECA":  "Europe, Russia & Central Asia",
    "WB_LAC": "Latin America & Caribbean",
    "WB_MENAP": "Middle East, N. Africa & Pakistan",
    "WB_NA": "North America",
    "WB_SA": "South Asia"
}

df_regions = df_aggregates[df_aggregates["code"].isin(wb_regions)].copy()
df_regions["region"] = df_regions["code"].map(region_labels)

fig_line = px.line(
    df_regions,
    x="year",
    y="researchers_per_million",
    color="region",
    markers=True,
    labels={
        "researchers_per_million": "Researchers per million",
        "year": "Year",
        "region": "Region"
    },
    title="R&D Researchers per Million People by World Region (1996–2023)"
)

fig_line.update_layout(
    height=500,
    hovermode="x unified",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    ),
    margin=dict(l=0, r=200, t=50, b=0)
)

fig_line.show()

### 3. The Most Underrepresented Countries

Which countries have the fewest researchers per million people?
The chart below shows the 20 countries with the lowest research capacity,
based on the most recent available data for each country.

Note: bars show the most recent available data point per country, which varies
significantly. Some values are 10–20 years old and should be interpreted with
caution — inconsistent reporting is itself a reflection of limited research
infrastructure in these countries.

**Absence is also data.** As of 2023, only 144 of the 193 UN member states
appear in this dataset — meaning roughly one in four countries has never
reported R&D data to UNESCO. The bottom 20 shown here are the bottom 20
among countries that report. The true bottom is invisible.

In [10]:
bottom20 = latest.nsmallest(20, "researchers_per_million").sort_values(
    "researchers_per_million"
)

fig_bar = px.bar(
    bottom20,
    x="researchers_per_million",
    y="country",
    orientation="h",
    color="researchers_per_million",
    color_continuous_scale="Reds_r",
    hover_data={"year": True, "researchers_per_million": ":.0f", "code": False},
    labels={
        "researchers_per_million": "Researchers per million",
        "country": "",
        "year": "Latest data year"
    },
    title="20 Countries with Fewest R&D Researchers per Million People"
)

fig_bar.update_layout(
    height=600,
    showlegend=False,
    coloraxis_showscale=False,
    margin=dict(l=0, r=50, t=50, b=0),
    xaxis=dict(range=[0, 120])
)

fig_bar.show()

In [11]:
print(bottom20)

                          country code  year  researchers_per_million
34   Democratic Republic of Congo  COD  2015                 10.10515
133                        Uganda  UGA  2023                 12.44448
53                      Guatemala  GTM  2021                 14.52588
69                           Laos  LAO  2002                 15.62884
3                          Angola  AGO  2016                 18.93579
127                      Tanzania  TZA  2013                 19.31703
36             Dominican Republic  DOM  2022                 20.03771
96                        Nigeria  NGA  2019                 22.12011
19                        Burundi  BDI  2018                 22.48160
71                        Lesotho  LSO  2015                 23.73201
95                          Niger  NER  2013                 26.93895
89                        Myanmar  MMR  2023                 28.00891
79                           Mali  MLI  2021                 28.66941
20                  

## Conclusion

The data paints a clear picture: research capacity is not a global resource —
it is heavily concentrated in a small number of high-income countries. The gap
between the top and bottom 10 countries is nearly 500x. Entire regions,
particularly Sub-Saharan Africa and South Asia, remain severely underrepresented
in the global research workforce.

This matters beyond fairness. Science that draws on a narrow slice of the world's
population produces knowledge shaped by that slice — its questions, assumptions,
and blind spots. Expanding access to research careers is not just an equity goal;
it is a prerequisite for more robust, globally relevant science.

---
*Data: UNESCO UIS Stat Bulk Data Download Service, via World Bank (2026),
processed by Our World in Data.*